# 02b · GasWindTunnel: R2 handcrafted features

**Purpose.** Compute k = 12 features per sensor (R2). Fit the 4 heads on both tasks with the same folds and seeds as 01a.

**Inputs**
- GasWindTunnel `X_signals.npy` (600, 72, 1500) + `meta.csv`

**Outputs**
- F_hand: (600, 72, 12) → (600, 864)
- `results/02b_gwt_handcrafted.csv`

In [1]:
import sys
sys.path.append("../..")          # project root, so that `import src` works

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("../..")
from src.cv import TASKS, load_baseline_level, load_processed, precomputed, run_cv, save_results, summarize
from src.features.handcrafted import FEATURE_NAMES, handcrafted_features
DATASET = "gwt"
REP = "R2_handcrafted"
NAME = "02b_gwt_handcrafted"          # results/02b_gwt_handcrafted.csv

X, meta, info = load_processed(DATASET)    # X: (N, C, T) baseline corrected
print(X.shape, meta["y_multi"].value_counts().sort_index().tolist())

(600, 72, 1500) [60, 60, 60, 60, 60, 60, 60, 60, 60, 60]


## R2: 12 handcrafted features per sensor
Nothing is fitted, so the features are computed once for all samples.

In [2]:
levels = load_baseline_level(DATASET)     # (N, C) resting level before baseline correction
F = handcrafted_features(X, levels, info["n_base"], info["exposure_end"])   # (N, C * 12) = (600, 864)
assert F.shape == (600, 864) and np.isfinite(F).all()
pd.DataFrame(F[:, :12], columns=FEATURE_NAMES).describe().T   # the 12 features of the first sensor

,count,mean,std,min,25%,50%,75%,max
baseline_level,600.0,472.854904,39.994859,413.922607,439.947922,459.020081,510.103416,638.325500
steady_state,600.0,-1.164386,2.175851,-9.751087,-2.414996,-1.115385,0.017595,23.424327
normalized,600.0,-0.002453,0.004423,-0.019983,-0.005062,-0.002434,0.000040,0.042971
peak,600.0,-1.283803,3.205477,-10.482025,-3.329758,-2.252960,1.929741,26.554504
time_to_peak,600.0,821.398333,373.206567,6.000000,509.750000,864.500000,1148.000000,1349.000000
rise_slope,600.0,-0.001054,0.007082,-0.011240,-0.003097,-0.001586,0.000265,0.118008
decay_slope,600.0,-0.002589,0.076023,-1.633240,-0.000901,0.000028,0.001422,0.326172
auc,600.0,-1370.165233,1956.506705,-9575.278870,-2544.778931,-1360.798035,-146.107521,17937.797119
ema_max_0.1,600.0,0.177587,0.012283,0.134350,0.170298,0.179454,0.186861,0.203029
ema_min_0.1,600.0,-0.105789,0.010969,-0.200000,-0.108662,-0.104196,-0.100214,-0.083123


## Cross validation: 4 heads × 5 folds × 3 seeds, both tasks
Everything that is fitted (scaler, heads, hyperparameters) sees the training fold only.

In [3]:
scores, preds = [], []
for task in TASKS:                                   # "multi", then "binary"
    s, p = run_cv(F, meta, DATASET, task, REP, precomputed)
    scores.append(s)
    preds.append(p)
scores, preds = pd.concat(scores, ignore_index=True), pd.concat(preds, ignore_index=True)
save_results(scores, preds, NAME)
print("saved results/" + NAME + ".csv")

gwt multi R2_handcrafted seed 0 fold 0: logreg 0.966  svm 0.950  rf 0.983  mlp 0.966


gwt multi R2_handcrafted seed 0 fold 1: logreg 1.000  svm 0.976  rf 1.000  mlp 0.992


gwt multi R2_handcrafted seed 0 fold 2: logreg 1.000  svm 0.992  rf 1.000  mlp 0.983


gwt multi R2_handcrafted seed 0 fold 3: logreg 0.992  svm 0.944  rf 1.000  mlp 0.975


gwt multi R2_handcrafted seed 0 fold 4: logreg 0.983  svm 0.961  rf 0.992  mlp 0.967


gwt multi R2_handcrafted seed 1 fold 0: logreg 1.000  svm 0.984  rf 1.000  mlp 0.992


... (20 outputs left out)

gwt binary R2_handcrafted seed 2 fold 1: logreg 0.954  svm 0.922  rf 1.000  mlp 0.954


gwt binary R2_handcrafted seed 2 fold 2: logreg 1.000  svm 0.950  rf 1.000  mlp 1.000


gwt binary R2_handcrafted seed 2 fold 3: logreg 1.000  svm 1.000  rf 1.000  mlp 1.000


gwt binary R2_handcrafted seed 2 fold 4: logreg 1.000  svm 1.000  rf 1.000  mlp 1.000
saved results/02b_gwt_handcrafted.csv


## Results: mean ± std over 3 seeds × 5 folds

In [4]:
print("macro F1"); display(summarize(scores))
print("balanced accuracy"); display(summarize(scores, "balanced_acc"))

macro F1


head                                  logreg            mlp             rf  \
dataset task   rep                                                           
gwt     binary R2_handcrafted  0.986 ± 0.018  0.984 ± 0.024  1.000 ± 0.000   
        multi  R2_handcrafted  0.991 ± 0.010  0.977 ± 0.012  0.996 ± 0.005   

head                                     svm  
dataset task   rep                            
gwt     binary R2_handcrafted  0.978 ± 0.027  
        multi  R2_handcrafted  0.963 ± 0.022

balanced accuracy


head                                  logreg            mlp             rf  \
dataset task   rep                                                           
gwt     binary R2_handcrafted  0.990 ± 0.017  0.982 ± 0.034  1.000 ± 0.000   
        multi  R2_handcrafted  0.991 ± 0.010  0.977 ± 0.012  0.996 ± 0.005   

head                                     svm  
dataset task   rep                            
gwt     binary R2_handcrafted  0.964 ± 0.044  
        multi  R2_handcrafted  0.961 ± 0.024

In [5]:
print("ROC AUC, binary task"); summarize(scores[scores["task"] == "binary"], "roc_auc")

ROC AUC, binary task


,,head,logreg,mlp,rf,svm
dataset,task,rep,,,,
gwt,binary,R2_handcrafted,0.999 ± 0.003,0.999 ± 0.003,1.000 ± 0.000,0.987 ± 0.021
